In [1]:
from typing import Tuple, Union
import numpy as np
from sympy import Matrix
from IPython.display import display

# Phân tách LU giải hệ tuyến tính

# Thuật toán
### 1. Phân tách LU
**Input**: $A_{n \times n}$

**Output**: $L$ và $U$

1. Khởi tạo $L=[0]_{n \times n}$ và $U = I_n$
2. Kiểm tra nếu $a_{11}=0 \to $ dừng chương trình, trái lại sang bước 3

3. Cho $i$ chạy từ $1 \to n$, gán: $\begin{cases} l_{i1}=a_{i1} \\[5pt] u_{1i}=\dfrac{a_{1i}}{l_{11}} \end{cases}$
4. Cho $i$ chạy từ $2 \to n$, thực hiện:

    4.1 Cho $j$ chạy từ $2 \to i$, gán: $l_{ij} = a_{ij} - \displaystyle\sum\limits_{k=1}^{j-1}l_{ik} \cdot u_{kj}$
    
    4.2 Kiểm tra nếu $l_{ii}=0 \to$ dừng chương trình, trái lại sang bước 4.3

    4.3 Cho $j$ chạy từ $i+1 \to n$, gán: $u_{ij} = \dfrac{a_{ij}-\displaystyle\sum\limits_{k=1}^{i-1}l_{ik} \cdot u_{kj}}{l_{ii}}$

5. In ra ma trận $L$ và $U$
### 2. Phân tách LU giải hệ $AX=B$
**Input**: $A_{n \times n}$ và $B_{n \times m}$

**Output**: Nghiệm $X$ của hệ phương trình

1. Tách $A$ thành $L, U$. Nếu không tách được, nhân 2 vế của $AX=B$ với $A^T$ và tách $M=A^T \cdot A$ thành $L,U$, gán $B=A^T \cdot B$
2. Khởi tạo $Y=[0]_{n \times m}$
3. Cho $j$ chạy từ $1 \to m$, gán: $y_{ij}=\dfrac{b_{1j}}{l_{11}}$
4. Cho $i$ chạy từ $2 \to n$, $j$ chạy từ $1 \to m$, gán: $y_{ij} = \dfrac{b_{ij}-\displaystyle\sum\limits_{k=1}^{i-1}l_{ik} \cdot y_{kj}}{l_{ii}}$
5. Cho $j$ chạy từ $1 \to m$, gán: $x_{nj} = \dfrac{y_{nj}}{u_{nn}}$
6. Cho $i$ chạy từ $n-1 \to 1$, $j$ chạy từ $1 \to m$, gán: $x_{ij} = \dfrac{y_{ij}-\displaystyle\sum\limits_{k=i+1}^{n}u_{ik} \cdot x_{kj}}{u_{ii}}$
7. In ra nghiệm $X$

In [ ]:
def output_matrix(X: np.ndarray, precision: int = 4):
    """
    Prints a NumPy array (vector or matrix) in LaTeX mathematical format using SymPy.
    
    Parameters:
    - X: np.ndarray, 1D or 2D array.
    - precision: number of decimal places to round floats to.
    """
    # Render natively as a LaTeX matrix in Jupyter
    display(Matrix(np.round(X, precision)))

In [ ]:
def lu_decomposition(A):
    """
    Phân tách LU cho ma trận A theo thuật toán (Câu 27).
    
    Parameters:
    A (numpy.ndarray): Input square matrix
    
    Returns:
    tuple: (L, U) matrices where U has diagonal elements = 1
    """
    n = len(A)
    L = np.zeros((n, n))
    U = np.zeros((n, n))
    
    # 1. Khởi tạo ma trận L = O, U = In
    np.fill_diagonal(U, 1)
    
    # 2. Kiểm tra A[1][1] = 0
    if A[0][0] == 0:
        raise ValueError("A[1][1] = 0, dừng thuật toán.")
        
    # 3. Base case (Cột 1 của L, Hàng 1 của U)
    for i in range(n):
        L[i][0] = A[i][0]
        U[0][i] = A[0][i] / L[0][0]

    print("L_1:")
    output_matrix(L[:, 0].reshape(-1, 1))
    print("U_1:")
    output_matrix(U[0, :].reshape(1, -1))
    
    # 4. Vòng lặp tổng quát (Tính theo hàng)
    for i in range(1, n):
        # 5.1: Tính ma trận L
        for j in range(1, i + 1):
            L[i][j] = A[i][j] - sum(L[i][k] * U[k][j] for k in range(j))
            
        # 5.2: Kiểm tra L[i][i] = 0
        if L[i][i] == 0:
            raise ValueError(f"L[{i+1}][{i+1}] = 0, kết thúc thuật toán.")
            
        # 5.3: Tính ma trận U
        for j in range(i + 1, n):
            U[i][j] = (A[i][j] - sum(L[i][k] * U[k][j] for k in range(i))) / L[i][i]

        print(f"L_{i + 1}:")
        output_matrix(L[i, :].reshape(1, -1))
        print(f"U_{i + 1}:")
        output_matrix(U[i, :].reshape(1, -1))
    
    return L, U

In [ ]:
def verify_lu_decomposition(Expected, L, U):
    """
    Verify LU decomposition by multiplying L and U and comparing with original matrix.
    
    Parameters:
    Expected (numpy.ndarray): Target matrix (e.g., M = A^T * A)
    L (numpy.ndarray): Lower triangular matrix
    U (numpy.ndarray): Upper triangular matrix
    """
    print("\nLower triangular matrix L:")
    output_matrix(L)
    print("\nUpper triangular matrix U:")
    output_matrix(U)
    
    # Multiply L and U
    LU = np.dot(L, U)
    
    # Check if LU equals Expected
    if np.allclose(LU, Expected):
        print("\nVerification successful: LU = M")
    else:
        print("\nVerification failed: LU ≠ M")

def lu_solve(L, U, C):
    """
    Giải hệ phương trình LUX = C bằng phép thế thuận và nghịch (Câu 28).
    
    Parameters:
    L (numpy.ndarray): Ma trận tam giác dưới
    U (numpy.ndarray): Ma trận tam giác trên
    C (numpy.ndarray): Ma trận vế phải
    
    Returns:
    numpy.ndarray: Ma trận nghiệm X
    """
    n = L.shape[0]
    cols = C.shape[1] if len(C.shape) > 1 else 1
    C_mat = C.reshape(n, cols)
    
    print("\n--- GIẢI HỆ PHƯƠNG TRÌNH LUX = C ---")
    print("Bước 1: Tính ma trận y (thế xuôi Ly = C)")
    
    # 2. Khởi tạo ma trận y
    y = np.zeros_like(C_mat, dtype=float)
    
    # 3. Base case cho y
    for j in range(cols):
        y[0, j] = C_mat[0, j] / L[0, 0]
    print("y_1:")
    output_matrix(y[0, :].reshape(1, -1))
        
    # 4. Vòng lặp tổng quát tính y (Quét theo hàng)
    for i in range(1, n):
        for j in range(cols):
            sum_val = sum(L[i, k] * y[k, j] for k in range(i))
            y[i, j] = (C_mat[i, j] - sum_val) / L[i, i]
        print(f"y_{i + 1}:")
        output_matrix(y[i, :].reshape(1, -1))
        
    print("\nMa trận y hoàn chỉnh:")
    output_matrix(y)
            
    print("\nBước 2: Tính ma trận X (thế ngược UX = y)")
    # 5. Base case cho X
    X = np.zeros_like(y, dtype=float)
    for j in range(cols):
        X[n - 1, j] = y[n - 1, j] / U[n - 1, n - 1]
    print(f"X_{n}:")
    output_matrix(X[n - 1, :].reshape(1, -1))
        
    # 6. Vòng lặp đi lùi tính X (Quét theo hàng)
    for i in range(n - 2, -1, -1):
        for j in range(cols):
            sum_val = sum(U[i, k] * X[k, j] for k in range(i + 1, n))
            X[i, j] = (y[i, j] - sum_val) / U[i, i]
        print(f"X_{i + 1}:")
        output_matrix(X[i, :].reshape(1, -1))
            
    return X

## Kết quả

In [ ]:
#Original matrix Ax=B
A_orig = np.loadtxt('A34.txt')
try:
    B_orig = np.loadtxt('B34.txt')
except OSError:
    B_orig = np.ones((A_orig.shape[0], 1)) # Fallback giả lập nếu chưa có file B

if len(B_orig.shape) == 1:
    B_orig = B_orig.reshape(-1, 1)

print("\nMa trận gốc A:"); output_matrix(A_orig)
print("\nMa trận gốc B:"); output_matrix(B_orig)

# Tính ma trận M = A^T * A
M = np.dot(A_orig.T, A_orig)
print("\nMa trận M = A^T * A (Dùng để phân rã LU):"); output_matrix(M)

# Tính ma trận C = A^T * B
C = np.dot(A_orig.T, B_orig)
print("\nMa trận C = A^T * B (Vế phải mới của hệ):"); output_matrix(C)


Ma trận gốc A:


Matrix([
[0.1588, 0.0064, 0.0025, 0.0304, 0.0014, 0.0083, 0.1594],
[0.0057, 0.2645, 0.0436, 0.0099, 0.0083, 0.0201, 0.3413],
[0.0264, 0.1506, 0.3557, 0.0139, 0.0142,  0.007, 0.0236],
[0.3299, 0.0565, 0.0495, 0.3636, 0.0204, 0.0483, 0.0649],
[0.0089, 0.0081, 0.0333, 0.0295, 0.3412, 0.0237,  0.002],
[ 0.119, 0.0901, 0.0996,  0.126, 0.1772, 0.2368, 0.3369],
[0.0063, 0.0126, 0.0196, 0.0098, 0.0064, 0.0132, 0.0012]])


Ma trận gốc B:


Matrix([
[ 74000.0],
[ 56000.0],
[ 10500.0],
[ 25000.0],
[ 17500.0],
[196000.0],
[  5000.0]])


Ma trận M = A^T * A (Dùng để phân rã LU):


Matrix([
[0.1491,  0.036, 0.0386, 0.1405, 0.0315,  0.046, 0.0894],
[ 0.036, 0.1042, 0.0774, 0.0372, 0.0243, 0.0308, 0.1289],
[0.0386, 0.0774, 0.1423, 0.0372, 0.0356, 0.0304, 0.0605],
[0.1405, 0.0372, 0.0372, 0.1503, 0.0402, 0.0488, 0.0747],
[0.0315, 0.0243, 0.0356, 0.0402, 0.1485, 0.0514, 0.0651],
[ 0.046, 0.0308, 0.0304, 0.0488, 0.0514, 0.0597, 0.0913],
[0.0894, 0.1289, 0.0605, 0.0747, 0.0651, 0.0913, 0.2602]])


Ma trận C = A^T * B (Vế phải mới của hệ):


Matrix([
[44106.35],
[36143.75],
[ 27801.3],
[ 37301.2],
[ 41961.7],
[49914.35],
[ 98852.1]])

In [ ]:
if M is not None:
    try:
        L, U = lu_decomposition(M)
        verify_lu_decomposition(M, L, U)
        
        # Giải hệ phương trình LUX = C
        X = lu_solve(L, U, C)
        
        print("\n=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:")
        print("Hệ có nghiệm duy nhất X =")
        output_matrix(X)
    except ValueError as e:
        print(f"Lỗi: {e}")

L_1:


Matrix([
[0.1491],
[ 0.036],
[0.0386],
[0.1405],
[0.0315],
[ 0.046],
[0.0894]])

U_1:


Matrix([[1.0, 0.2416, 0.2592, 0.9427, 0.2116, 0.3088, 0.5998]])

L_2:


...

Matrix([[-181542.3864]])

X_1:


Matrix([[218957.2299]])


=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:
Hệ có nghiệm duy nhất X =


Matrix([
[ 218957.2299],
[-181542.3864],
[  70561.5994],
[-223624.7408],
[  28059.7496],
[ 470660.1697],
[ 270184.9952]])

In [ ]:
if 'A_orig' in locals() and A_orig is not None:
    print("\n--- PHÂN RÃ LU CHO TRỰC TIẾP MA TRẬN A ---")
    try:
        L_A, U_A = lu_decomposition(A_orig)
        verify_lu_decomposition(A_orig, L_A, U_A)
    except ValueError as e:
        print(f"Lỗi: {e}")


--- PHÂN RÃ LU CHO TRỰC TIẾP MA TRẬN A ---
L_1:


Matrix([
[0.1588],
[0.0057],
[0.0264],
[0.3299],
[0.0089],
[ 0.119],
[0.0063]])

U_1:


Matrix([[1.0, 0.0403, 0.0157, 0.1914, 0.0088, 0.0523, 1.0038]])

L_2:


...


Lower triangular matrix L:


Matrix([
[0.1588,    0.0,    0.0,    0.0,    0.0,    0.0,     0.0],
[0.0057, 0.2643,    0.0,    0.0,    0.0,    0.0,     0.0],
[0.0264, 0.1495, 0.3307,    0.0,    0.0,    0.0,     0.0],
[0.3299, 0.0432, 0.0372, 0.2986,    0.0,    0.0,     0.0],
[0.0089, 0.0077, 0.0319, 0.0272, 0.3386,    0.0,     0.0],
[ 0.119, 0.0853, 0.0837, 0.0994, 0.1661,  0.206,     0.0],
[0.0063, 0.0123, 0.0175,  0.008, 0.0051, 0.0112, -0.0162]])


Upper triangular matrix U:


Matrix([
[1.0, 0.0403, 0.0157, 0.1914, 0.0088,  0.0523,  1.0038],
[0.0,    1.0, 0.1646, 0.0333, 0.0312,  0.0749,  1.2698],
[0.0,    0.0,    1.0, 0.0117, 0.0281, -0.0169,  -0.583],
[0.0,    0.0,    0.0,    1.0, 0.0506,  0.0953, -1.0029],
[0.0,    0.0,    0.0,    0.0,    1.0,  0.0609,  0.0858],
[0.0,    0.0,    0.0,    0.0,    0.0,     1.0,  1.1811],
[0.0,    0.0,    0.0,    0.0,    0.0,     0.0,     1.0]])


Verification successful: LU = M
